# Harry Potter RAG — Lab 2

Continuing from last lab, we are using the famous *Harry Potter & The Philosopher's Stone* by J.K. Rowling, including:

- Text processing
- Chunking
- Embedding
- VectorDB management
- Q&A
- Evaluation

This time we are focusing on evaluation of the effectiveness of retrieval & generation step, by altering how we prepare the chunks of data.

*Disclaimer: The source book is copyrighted and is not included in this repository. This lab is for personal, non-commercial study only — only original code, questions, and metrics are published. AI coding agent is used in the process to assist notes writing and project planning.*

## Basic Setup (Reusing Lab 1)
- Import dependencies
- Setup env variables
- Setup Bedrock client
- Connect local chromadb

In [4]:
import boto3
from botocore.config import Config
import pymupdf
import re
import json
import os
import hashlib
from pathlib import Path
import tiktoken
import numpy as np
from tqdm.auto import tqdm
import chromadb
from dotenv import load_dotenv
from rank_bm25 import BM25Okapi

def repo_root() -> Path:
    """Walk up from the cwd until the repo root (the dir containing .git)."""
    p = Path.cwd()
    while p != p.parent and not (p / ".git").exists():
        p = p.parent
    return p

ROOT = repo_root()
LABS = ROOT / "labs"
DATA = LABS / "data"

load_dotenv(dotenv_path=str(LABS / ".env"))
os.getenv("AWS_REGION")

class Bedrock:
    """Bedrock helpers: Converse for chat, Titan for embeddings (sequential + cached)."""

    def __init__(self, model_id=None, region=None, embedding_model_id=None):
        self.region = region or os.getenv("AWS_REGION") or "ap-northeast-1"
        self.model_id = model_id or os.getenv("BEDROCK_CONVERSATION_MODEL_ID")
        if not self.model_id:
            raise ValueError("Set BEDROCK_CONVERSATION_MODEL_ID in .env")
        self.embedding_model_id = embedding_model_id or os.getenv("BEDROCK_EMBEDDING_MODEL_ID")
        if not self.embedding_model_id:
            raise ValueError("Set BEDROCK_EMBEDDING_MODEL_ID in .env")
        self.client = boto3.client(
            "bedrock-runtime",
            region_name=self.region,
            config=Config(read_timeout=120, retries={"max_attempts": 5, "mode": "standard"}),
        )

    # ---- conversation (Q&A and judge): override model/params per call ----
    def chat(self, prompt, system=None, model_id=None, temperature=0.0, max_tokens=1024):
        kwargs = {
            "modelId": model_id or self.model_id,
            "messages": [{"role": "user", "content": [{"text": prompt}]}],
            "inferenceConfig": {"temperature": temperature, "maxTokens": max_tokens},
        }
        if system:
            kwargs["system"] = [{"text": system}]
        response = self.client.converse(**kwargs)
        return response["output"]["message"]["content"][0]["text"]

    # ---- embeddings: sequential + disk cache ----
    @staticmethod
    def _cache_path():
        return DATA / "embeddings_cache.jsonl"

    @staticmethod
    def _cache_key(text, model_id, dimensions, normalize):
        raw = f"{model_id}|{dimensions}|{normalize}|{text}"
        return hashlib.sha256(raw.encode("utf-8")).hexdigest()

    def _embed_one(self, text, dimensions, normalize):
        body = json.dumps({"inputText": text, "dimensions": dimensions, "normalize": normalize})
        resp = self.client.invoke_model(modelId=self.embedding_model_id, body=body)
        return json.loads(resp["body"].read())["embedding"]

    def embed(self, texts, dimensions=1024, normalize=True, cache_path=None):
        """Embed texts one by one, appending results to a JSONL cache.

        Returns an (n, dimensions) float32 array. 
        Re-running the notebook will proceed search for already-cached texts, 
        those will make no API calls.
        """
        if isinstance(texts, str):
            texts = [texts]
        texts = list(texts)
        cache_path = Path(cache_path) if cache_path else self._cache_path()

        cache = {}
        if cache_path.exists():
            for line in cache_path.read_text(encoding="utf-8").splitlines():
                if line.strip():
                    rec = json.loads(line)
                    cache[rec["key"]] = rec["vector"]

        model = self.embedding_model_id
        keys = [self._cache_key(t, model, dimensions, normalize) for t in texts]
        todo = [(i, t) for i, t in enumerate(texts) if keys[i] not in cache]

        if todo:
            cache_path.parent.mkdir(parents=True, exist_ok=True)
            with open(cache_path, "a", encoding="utf-8") as f:

                # Process bar
                for i, t in tqdm(todo, desc="embedding"):
                    vec = self._embed_one(t, dimensions, normalize)
                    cache[keys[i]] = vec
                    f.write(json.dumps({"key": keys[i], "vector": vec}) + "\n")

        return np.array([cache[k] for k in keys], dtype=np.float32)


bedrock_client = Bedrock()
judge_client = Bedrock(model_id=os.getenv("BEDROCK_JUDGE_MODEL_ID"))

chroma_client = chromadb.PersistentClient(path=str(LABS / ".chroma"))


## Goal of Lab 2: RAG Evaluation & Ablation Matrix

In this lab, we take our baseline pipeline and measure how different design choices actually move the numbers. Instead of treating RAG like a black box, we split evaluation into two separate questions: 
1. Did we retrieve the right context?
2. Did the LLM generate a correct, grounded answer?

### What We're Testing

1. Ground-Truth Evaluation 
   We load our hand-written Q&A test set containing exact text evidence snippets, multi-chapter questions, and out-of-scope queries designed to test refusal behavior.

2. Ablation Grid
   We run an automated matrix comparison across two main knobs:
   - **Chunking:** Fixed-size (256 vs. 512 tokens) vs. Chapter-aware boundaries.
   - **Retrieval:** Dense (Titan V2 embeddings), Sparse (BM25), and Hybrid (Reciprocal Rank Fusion).

3. Separating the Failures
   - **Retriever metrics:** We measure `Hit@k`, `Recall@k`, and `MRR` to see if the gold evidence made it into the context window.
   - **Generator metrics:** We use an LLM-as-a-Judge to check answer correctness, faithfulness (preventing hallucinations), and false-abstention rates.

4. Results Leaderboard  
   Every run logs directly to `eval/results.csv`, producing a final comparison table so we can back up architectural choices with actual numbers.